## Importing Libraries

In [28]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import GroupShuffleSplit

from tqdm.auto import tqdm

import warnings
warnings.filterwarnings("ignore")

## Data Loading Normalize Prompt

In [29]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

def normalize_prompt(text):

    text = text.lower()

    text = text.replace("based on the given context", "")
    text = text.replace("carefully", "")
    text = text.replace("among the listed options", "")
    text = text.replace("from the following choices", "")

    return " ".join(text.split())


train_df["group"] = train_df["prompt"].apply(normalize_prompt)
test_df["group"] = test_df["prompt"].apply(normalize_prompt)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer,group
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer: what is martin ...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion?
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option: what is the term...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option: what is marti...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement: what is the co...


## Data Into Pairwise Dataset

In [30]:
def make_pairwise(df, train=True):

    rows = []

    for _, row in df.iterrows():

        for option in ["A", "B", "C", "D", "E"]:

            rows.append({

                "id": row["id"],

                "group": row["group"],

                "text":
                f"Question: {row['prompt']} [SEP] Option: {row[option]}",

                "option": option,

                "label":
                int(option == row["answer"]) if train else -1

            })

    return pd.DataFrame(rows)


pair_train = make_pairwise(train_df, train=True)
pair_test = make_pairwise(test_df, train=False)

print(pair_train.shape)
print(pair_test.shape)

pair_train.head()

(10000, 5)
(2500, 5)


,id,group,text,option,label
0,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,A,0
1,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,B,1
2,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,C,0
3,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,D,0
4,1,pick the best possible answer: what is martin ...,Question: Pick the best possible answer: What ...,E,0


## Tokenizer and Train Validation Split

In [31]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

groups = pair_train["group"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(
        pair_train,
        pair_train["label"],
        groups
    )
)

train_pair = pair_train.iloc[train_idx].reset_index(drop=True)
valid_pair = pair_train.iloc[valid_idx].reset_index(drop=True)

print(train_pair.shape)
print(valid_pair.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

(7995, 5)
(2005, 5)


## Dataset

In [32]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len=128):

        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        text = self.data.iloc[idx]["text"]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_attention_mask=True
        )

        item = {
            "input_ids": torch.tensor(
                encoding["input_ids"],
                dtype=torch.long
            ),

            "attention_mask": torch.tensor(
                encoding["attention_mask"],
                dtype=torch.long
            )
        }

        if "label" in self.data.columns:
            item["labels"] = torch.tensor(
                self.data.iloc[idx]["label"],
                dtype=torch.float
            )

        return item

## DataLoader

In [33]:
train_dataset = MCQDataset(
    train_pair,
    tokenizer
)

valid_dataset = MCQDataset(
    valid_pair,
    tokenizer
)

test_dataset = MCQDataset(
    pair_test,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(len(train_loader))
print(len(valid_loader))
print(len(test_loader))

500
63
79


In [34]:
batch = next(iter(train_loader))

# print(batch["input_ids"].shape) //extra
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 128])
torch.Size([16])


## Model

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BertRanker(nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.2)

        self.fc = nn.Linear(
            self.bert.config.hidden_size,
            1
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0]

        cls = self.dropout(cls)
        # cls = cls.float()

        score = self.fc(cls)

        return score.squeeze(-1)


model = BertRanker().to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

criterion = nn.BCEWithLogitsLoss()

print("Model Loaded")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded


In [36]:
# extra
batch = next(iter(train_loader))

with torch.no_grad():

    logits = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )

print(logits[:5])

print(torch.isnan(logits).any()) 

tensor([-0.3812, -0.0502,  0.0932, -0.4429, -0.0320], device='cuda:0')
tensor(False, device='cuda:0')


## Training Loop

In [37]:
epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids,
            attention_mask
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        loop.set_description(
            f"Epoch {epoch+1}"
        )

        loop.set_postfix(
            loss=loss.item()
        )

    print(
        f"Epoch {epoch+1} Loss : {total_loss/len(train_loader):.4f}"
    )

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch 1 Loss : 0.4901


  0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Validation Predictions

In [ ]:
model.eval()

valid_scores = []

with torch.no_grad():

    for batch in tqdm(valid_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(
            input_ids,
            attention_mask
        )

        scores = torch.sigmoid(logits)

        valid_scores.extend(
            scores.cpu().numpy()
        )

valid_scores = np.array(valid_scores)

print(valid_scores[:10])

## MAP@3 and Accuracy Score

In [ ]:
valid_df = valid_pair.copy()
valid_df["score"] = valid_scores

predictions = []
true_answers = []

for _, group in valid_df.groupby("id"):

    group = group.sort_values("score", ascending=False)

    predictions.append(group["option"].head(3).tolist())

    true_answers.append(
        group[group["label"] == 1]["option"].values[0]
    )


def map3(y_true, y_pred):

    score = 0

    for t, p in zip(y_true, y_pred):

        if t == p[0]:
            score += 1

        elif t == p[1]:
            score += 0.5

        elif t == p[2]:
            score += 1/3

    return score / len(y_true)


acc = np.mean(
    [t == p[0] for t, p in zip(true_answers, predictions)]
)

print("Validation MAP@3 :", map3(true_answers, predictions))
print("Validation Accuracy :", acc)

## Test Prediction

In [ ]:
model.eval()

test_scores = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(
            input_ids,
            attention_mask
        )

        scores = torch.sigmoid(logits)

        test_scores.extend(
            scores.cpu().numpy()
        )

test_scores = np.array(test_scores)

## submission.csv

In [ ]:
test_result = pair_test.copy()

test_result["score"] = test_scores

predictions = []

for _, group in test_result.groupby("id"):

    group = group.sort_values("score", ascending=False)

    predictions.append(
        " ".join(group["option"].head(3).tolist())
    )

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())